# SLM Fine-Tuning for RAG - Kaggle Notebook

**Fine-tuning Qwen3-1.7B with QLoRA for RAG on Italian Public Administration Documents**

This notebook is adapted for Kaggle and uses your dataset at:
`/kaggle/input/datasets/marcolil/empulia-regulations`

## Prerequisites
1. Kaggle Notebook runtime with GPU (T4 recommended)
2. Project code available in `/kaggle/working/SLM_Finetuning_4_RAG` or current working directory
3. Dataset attached to the notebook

## 1. Kaggle Environment Setup

In [ ]:
from pathlib import Path
import os
import platform

IS_KAGGLE = Path('/kaggle').exists()
print(f'Running on Kaggle: {IS_KAGGLE}')
print(f'Python: {platform.python_version()}')
print(f'Current working directory: {Path.cwd()}')

if not IS_KAGGLE:
    print('Warning: this notebook is optimized for Kaggle paths.')

In [ ]:
from pathlib import Path
import os
import sys

DATASET_PATH = Path('/kaggle/input/datasets/marcolil/empulia-regulations')

# Auto-detect project root
candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/kaggle/working/SLM_Finetuning_4_RAG'),
]

PROJECT_PATH = None
for candidate in candidates:
    if (candidate / 'src').exists() and (candidate / 'requirements.txt').exists():
        PROJECT_PATH = candidate.resolve()
        break

if PROJECT_PATH is None:
    raise FileNotFoundError(
        'Project root not found. Ensure src/ and requirements.txt are available in Kaggle working directory.'
    )

os.chdir(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))
print(f'Project path: {PROJECT_PATH}')
print(f'Dataset path exists: {DATASET_PATH.exists()}')
print('Project files:', sorted([p.name for p in PROJECT_PATH.iterdir()])[:10])

In [ ]:
# Install dependencies from project requirements
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Sync Kaggle Dataset and Preprocess

This section copies the 16 Empulia regulation PDFs from Kaggle input into the project `data/` folder, then runs preprocessing if outputs are missing.

In [ ]:
from pathlib import Path
import shutil
import importlib.util

data_dir = PROJECT_PATH / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(DATASET_PATH.glob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDF files found in {DATASET_PATH}')

copied = 0
for pdf in pdf_files:
    target = data_dir / pdf.name
    if not target.exists():
        shutil.copy2(pdf, target)
        copied += 1

print(f'PDFs available in Kaggle dataset: {len(pdf_files)}')
print(f'Newly copied PDFs: {copied}')
print(f'Total PDFs in project data dir: {len(list(data_dir.glob("*.pdf")))}')

sft_dataset_path = PROJECT_PATH / 'outputs' / 'qa_dataset' / 'sft_dataset'
if sft_dataset_path.exists():
    print('Preprocessing already available. Skipping preprocess step.')
else:
    data_pkg_exists = importlib.util.find_spec('src.data') is not None
    if not data_pkg_exists:
        print('Warning: src.data package is missing, so preprocess cannot run in this workspace.')
        print('You can still continue if outputs/qa_dataset/sft_dataset was generated previously.')
    else:
        print('Running preprocess pipeline...')
        !python -m src.main preprocess

## 3. QLoRA Fine-Tuning

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_PATH))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

from src.config import get_config
from src.training.train import train_model

config = get_config()

# On T4 (16GB VRAM) we can use slightly larger batch size
config.training.per_device_train_batch_size = 2
config.training.gradient_accumulation_steps = 4
config.training.num_train_epochs = 3

print(f'Training config:')
print(f'  Batch size: {config.training.per_device_train_batch_size}')
print(f'  Gradient accumulation: {config.training.gradient_accumulation_steps}')
print(f'  Effective batch: {config.training.per_device_train_batch_size * config.training.gradient_accumulation_steps}')
print(f'  Epochs: {config.training.num_train_epochs}')
print(f'  Learning rate: {config.training.learning_rate}')
print(f'  LoRA r: {config.lora.r}, alpha: {config.lora.lora_alpha}')

In [ ]:
# Run training
metrics = train_model(config)

print('\n' + '='*60)
print('Training Complete!')
print('='*60)
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 4. Build Vector Index (if not done)

In [ ]:
from pathlib import Path
from src.config import get_config

config = get_config()
if not (config.paths.vectorstore_dir / 'index.faiss').exists():
    print('Building FAISS index...')
    from src.rag.vectorstore import build_vectorstore
    vs = build_vectorstore(config)
    print('Done!')
else:
    print('FAISS index already exists!')

## 5. Evaluation

In [ ]:
from src.evaluation.evaluate import evaluate_rag
from src.config import get_config
import json

config = get_config()

# Evaluate base model
print('Evaluating BASE model...')
base_scores = evaluate_rag(config, use_finetuned=False)
print(f'Base scores: {json.dumps(base_scores["aggregate_scores"], indent=2)}')

print('\n' + '='*60)

# Evaluate fine-tuned model
print('Evaluating FINE-TUNED model...')
ft_scores = evaluate_rag(config, use_finetuned=True)
print(f'Fine-tuned scores: {json.dumps(ft_scores["aggregate_scores"], indent=2)}')

## 6. Comparison and Plots

In [ ]:
from src.evaluation.compare import run_comparison
from src.config import get_config

config = get_config()
summary = run_comparison(config)

# Display plots inline
from IPython.display import Image, display
from pathlib import Path

plots_dir = config.paths.plots_dir
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f'\n--- {plot_file.name} ---')
    display(Image(filename=str(plot_file), width=700))

## 7. Interactive Demo

In [ ]:
from src.rag.pipeline import RAGPipeline
from src.config import get_config

config = get_config()

# Load fine-tuned pipeline
pipeline = RAGPipeline(config=config, use_finetuned=True)

# Test questions
test_questions = [
    'Quali sono le procedure previste per gli appalti pubblici?',
    'Cosa prevede la normativa sulle firme elettroniche?',
    'Quali sono gli obblighi delle stazioni appaltanti?',
    'Come funziona il sistema di qualificazione delle imprese?',
]

for q in test_questions:
    result = pipeline.query(q)
    print(f'\n{"="*60}')
    print(f'Q: {result["question"]}')
    print(f'A: {result["answer"][:300]}...')
    print(f'Sources: {result["source_documents"]}')
    print(f'Time: {result["total_time_s"]}s')

pipeline.cleanup()

## 8. Export Results

Create a ZIP archive under `/kaggle/working` so you can download artifacts after execution.

In [ ]:
# Zip outputs for Kaggle download
import shutil
from pathlib import Path

archive_base = Path('/kaggle/working/slm_rag_outputs')
archive_path = shutil.make_archive(str(archive_base), 'zip', str(PROJECT_PATH / 'outputs'))
print(f'Archive created: {archive_path}')
print('Open the right panel in Kaggle (Output) to download the zip file.')